In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix,accuracy_score,precision_score,
                             recall_score,f1_score,roc_auc_score)
from sklearn.base import clone
from sklearn.pipeline import Pipeline         
from sklearn.svm import SVC                    
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df=pd.read_csv(r"C:\Users\USER\Desktop\5thsem_OJT\project1_diabeties\data\prosessed\5_6271303059964437340.csv")

feature_cols = ["age","pregnancies","glucose","blood_pressure",
                "skin_thickness","insulin","bmi","diabetes_pedigree",
                "bmi_category","glucose_band"]


x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]


x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)


candidates = {
    "LogReg": ("scaled", LogisticRegression(max_iter=1000)),
    "Decision Tree": ("raw", DecisionTreeClassifier(max_depth=3, random_state=42)),
    "KNN": ("scaled", KNeighborsClassifier(n_neighbors=25)),
    "SVC balanced": ("scaled",SVC(class_weight="balanced", random_state=42)),
    "RF balanced": ("raw", RandomForestClassifier(
        max_depth=5, class_weight="balanced", random_state=42
    )),
}

TARGET_RECALL, GUARD_F1 = 0.70, 0.50   

print("rows:", len(df), "| train:", len(x_train), "| test:", len(x_test))
print("diabetic in test set:", int(y_test.sum()))
print("features after get_dummies:", x.shape[1])
print("candidates:", list(candidates))
print("target: recall >=", TARGET_RECALL, "with F1 >=", GUARD_F1)



rows: 950 | train: 760 | test: 190
diabetic in test set: 55
features after get_dummies: 13
candidates: ['LogReg', 'Decision Tree', 'KNN', 'SVC balanced', 'RF balanced']
target: recall >= 0.7 with F1 >= 0.5


In [ ]:
df.columns

Index(['patient_id', 'age', 'gender', 'pregnancies', 'glucose',
       'blood_pressure', 'skin_thickness', 'insulin', 'bmi',
       'diabetes_pedigree', 'physical_activity', 'smoker', 'last_checkup_date',
       'outcome', 'age_band', 'bmi_category', 'glucose_band',
       'insulin_glucose', 'pregnancies_rate', 'since_checkup',
       'activity_smoker'],
      dtype='str')

# re split the data and watch the score moved


In [ ]:
rows=[]

for rs in [0,1,7,42,99]:
    xa,xb,ya,yb = train_test_split(x,y,test_size=0.2,
    random_state=rs,stratify=y)
    
    row = {"split":rs}
    for name, (kind,model) in candidates.items():
        m=clone(model)
        if kind =="scaled":
            sc =StandardScaler().fit(xa)
            m.fit(sc.transform(xa),ya)
            pred = m.predict(sc.transform(xb))
        else:
            m.fit(xa,ya)
            pred =m.predict(xb)
        row[name]= round(recall_score(yb,pred),3)
    rows.append(row)      


splits = pd.DataFrame(rows).set_index("split")
print(splits)
spread = (splits.max() - splits.min()).round(3)
print("\nspread (best - worst):\n", spread.sort_values(ascending=False))    

        

       LogReg  Decision Tree    KNN  SVC balanced  RF balanced
split                                                         
0       0.218          0.091  0.127         0.600        0.600
1       0.255          0.382  0.164         0.745        0.618
7       0.255          0.182  0.200         0.655        0.582
42      0.291          0.291  0.182         0.618        0.655
99      0.273          0.236  0.200         0.709        0.673

spread (best - worst):
 Decision Tree    0.291
SVC balanced     0.145
RF balanced      0.091
LogReg           0.073
KNN              0.073
dtype: float64


**K-fold cross-validation** is a way to check how well a model may perform on new data.

1. Split the data into **K groups**, called folds.
2. Train the model using all but one fold.
3. Test it on the fold left out.
4. Repeat until each fold has been used for testing.
5. Average the scores from all rounds.

For example, with **5-fold cross-validation**, the model is trained and tested five times, each time using a different 20% of the data for testing.

# cross validation
**setup cross validation the right way**

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)

for i, (tr, te) in enumerate(cv.split(x_train, y_train), 1):
    print(f"fold {i}: train {len(tr)} score {len(te)} "
          f"diabetic in score fold {y_train.iloc[te].sum()}")

def make_model(kind, model):
    """Scaled models get the scaler INSIDE a Pipeline, so it refits per fold."""
    if kind == "scaled":
        return Pipeline([("scale", StandardScaler()), ("model", model)])
    return model

print()
for name, (kind, model) in candidates.items():
    print(f"{name:15s} -> {type(make_model(kind, model)).__name__}")

fold 1: train 608 score 152 diabetic in score fold 43
fold 2: train 608 score 152 diabetic in score fold 43
fold 3: train 608 score 152 diabetic in score fold 44
fold 4: train 608 score 152 diabetic in score fold 44
fold 5: train 608 score 152 diabetic in score fold 44

LogReg          -> Pipeline
Decision Tree   -> DecisionTreeClassifier
KNN             -> Pipeline
SVC balanced    -> Pipeline
RF balanced     -> RandomForestClassifier


# **cross validate every candidate**

In [ ]:
from sklearn.model_selection import cross_validate

rows = []
for name, (kind, model) in candidates.items():
  est = make_model(kind, model)
  res = cross_validate(
      est,x_train,y_train,cv=cv,scoring=["recall", "f1"],
  )
  single = clone(est).fit(x_train, y_train)
  rows.append({
      "model": name,
      "cv_recall": round(res["test_recall"].mean(), 3),
      "cv_std": round(res["test_recall"].std(), 3),
      "cv_f1": round(res["test_f1"].mean(), 3),
      "week6_recall":round(recall_score(y_test,single.predict(x_test)),3),
      "folds": np.round(res["test_recall"],3).tolist(),

  })
cv_table = pd.DataFrame(rows).sort_values("cv_recall",ascending=False)
print(cv_table.to_string(index=False))

        model  cv_recall  cv_std  cv_f1  week6_recall                               folds
 SVC balanced      0.721   0.046  0.559         0.618 [0.744, 0.791, 0.682, 0.727, 0.659]
  RF balanced      0.652   0.051  0.521         0.655 [0.721, 0.651, 0.682, 0.636, 0.568]
       LogReg      0.303   0.073  0.390         0.291  [0.419, 0.326, 0.318, 0.25, 0.205]
Decision Tree      0.216   0.076  0.296         0.291 [0.116, 0.326, 0.159, 0.205, 0.273]
          KNN      0.174   0.018  0.261         0.182 [0.186, 0.163, 0.159, 0.205, 0.159]


In [ ]:
cv_table ['low'] = (cv_table['cv_recall'] - cv_table['cv_std']).round(3)
cv_table ['high'] = (cv_table['cv_recall'] + cv_table['cv_std']).round(3)
cv_table['clears_target'] = cv_table['cv_recall'] >= TARGET_RECALL
cv_table['clears_guard'] = cv_table['cv_f1'] >= GUARD_F1

print(cv_table[['model','cv_recall','low','high','cv_f1','clears_target','clears_guard']].to_string(index=False))

        model  cv_recall   low  high  cv_f1  clears_target  clears_guard
 SVC balanced      0.721 0.675 0.767  0.559           True          True
  RF balanced      0.652 0.601 0.703  0.521          False          True
       LogReg      0.303 0.230 0.376  0.390          False         False
Decision Tree      0.216 0.140 0.292  0.296          False         False
          KNN      0.174 0.156 0.192  0.261          False         False


## Hyperparameter Tuning 

Hyperparameter Tuning
The process of finding the best settings for a machine learning model to improve its performance.

GridSearchCV
A hyperparameter tuning technique that tests every specified combination of hyperparameter values using cross-validation.

RandomizedSearchCV
A hyperparameter tuning technique that tests a fixed number of randomly selected hyperparameter combinations using cross-validation.

Cross-Validation (CV)
A technique that evaluates a model using different portions of the training data.

**** 

Hyperparameter tuning finds the model settings that give the best results.

- **GridSearchCV** tests every combination of settings you provide. It is thorough but can be slow.
- **RandomizedSearchCV** tests a chosen number of random combinations. It is faster for large searches.

Both use cross-validation to compare settings across multiple data splits.

**Use GridSearchCV** for a small set of options.


**Use RandomizedSearchCV** for many options or when you need a faster search.

****


**Hyperparameter tuning is the process of finding the best settings for a Machine Learning model to improve its performance and accuracy**
1. What is GridSearchCV?

 GridSearchCV is a hyperparameter tuning technique that tests all specified combinations of hyperparameter values to find the best settings for a Machine Learning model.
 
 2. What is RandomizedSearchCV?

 RandomizedSearchCV is a hyperparameter tuning technique that tests a limited number of randomly selected combinations from the specified hyperparameter options or distributions.

# Difference Between Parameter and Hyperparameter

**Parameter:** A value learned automatically by a machine learning model during training. Example: weights and bias in Linear Regression.

**Hyperparameter:** A setting chosen before training that controls how the model learns. Example: `n_neighbors` in KNN and `max_depth` in Decision Tree.

| Parameter | Hyperparameter |
|---|---|
| Learned during training. | Set before training. |
| Updated by the model during learning. | Usually selected by the user or tuning methods. |
| Example: Weights and bias. | Example: Learning rate, `n_neighbors`. |


******
- Parameters are learned by the model during training.  
  *Example:* the weights in a linear regression model.

- Hyperparameters are settings chosen before training.  
  *Example:* the tree depth in a decision tree or the learning rate.

In short: the model learns parameters; you choose hyperparameters.```

In [ ]:
rf = RandomForestClassifier(random_state=42)
for setting, value in rf.get_params().items():
    print(f'{setting:25s} {value}')

bootstrap                 True
ccp_alpha                 0.0
class_weight              None
criterion                 gini
max_depth                 None
max_features              sqrt
max_leaf_nodes            None
max_samples               None
min_impurity_decrease     0.0
min_samples_leaf          1
min_samples_split         2
min_weight_fraction_leaf  0.0
monotonic_cst             None
n_estimators              100
n_jobs                    None
oob_score                 False
random_state              42
verbose                   0
warm_start                False


# **Build the Grid and count the cost**

In [ ]:
param_grid = {
    'max_depth' : [3, 5, 8, None],
    'min_samples_leaf' : [1, 5, 10],
    'class_weight' : ['balanced','balanced_subsample']
}

# count all combination 
combos = 1

for values in param_grid.values():
    combos *= len(values)

# Calculate total model fits  
folds = cv.get_n_splits()
totals_fits = combos * folds 

print('Combination            :', combos)
print('Cross-validation folds :', folds)
print('Total model fits       :', totals_fits)



Combination            : 24
Cross-validation folds : 5
Total model fits       : 120


# **Run grid search on the leading forest**

In [ ]:
from sklearn.model_selection import GridSearchCV

# Create the grid search
gs_recall = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring='recall',
    n_jobs=-1,
    
    )

# Run the search 
gs_recall.fit(x_train, y_train)

#Display the best settings and recall 
print('Best setings   :', gs_recall.best_params_)
print('Best CV recall :', round(gs_recall.best_score_,3))

# display the top 5 combination 
top = pd.DataFrame(gs_recall.cv_results_)

print(
    top[['params','mean_test_score','std_test_score','rank_test_score']]
    .sort_values('rank_test_score')
    .head(5)
    .to_string(index=False)
)

Best setings   : {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}
Best CV recall : 0.739
                                                                        params  mean_test_score  std_test_score  rank_test_score
           {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}         0.738689        0.041181                1
           {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 5}         0.738584        0.041746                2
          {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 10}         0.734249        0.062548                3
{'class_weight': 'balanced_subsample', 'max_depth': 3, 'min_samples_leaf': 10}         0.720719        0.064830                4
 {'class_weight': 'balanced_subsample', 'max_depth': 3, 'min_samples_leaf': 5}         0.706871        0.065250                5


# Test the tuned model on unseen patients 


In [ ]:
best_rf = gs_recall.best_estimator_

#predict for unseen test patients 
pred = best_rf.predict(x_test)

# calculate confusion matrix value 
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

#compare cross-validation and test performance 
print('Search CV recall :', round(gs_recall.best_score_,3))
print('Test recall      :', round(recall_score(y_test,pred),3))
print('Test F1          :', round(f1_score(y_test,pred),3))

print(f'Diabetic patients found : {tp} of {tp + fn}')
print(f'Missed patients         : {fn}')

Search CV recall : 0.739
Test recall      : 0.745
Test F1          : 0.536
Diabetic patients found : 41 of 55
Missed patients         : 14


# change the scoring , change the winner 

In [ ]:
from sklearn.model_selection import GridSearchCV

# Create the grid search
gs_f1 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    
    )

# Run the search 
gs_f1.fit(x_train, y_train)

#compare both tuned models on unseen test patients 
for label, gs in [
    ('Searched on recall', gs_recall),
    ('Searched on F1', gs_f1)    
]:
    pred = gs.best_estimator_.predict(x_test)

    found = int(((pred == 1) & (y_test.values == 1)).sum())

    print(label, ':', gs.best_params_)
    print('Test recall    :', round(recall_score(y_test, pred),3))
    print('Test F1        :', round(f1_score(y_test, pred),3))
    print('Patients found :', found)

Searched on recall : {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}
Test recall    : 0.745
Test F1        : 0.536
Patients found : 41
Searched on F1 : {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_leaf': 10}
Test recall    : 0.655
Test F1        : 0.522
Patients found : 36


# Tune the runner-up with a random search 


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Create SVM pipeline 

svc_pipe = Pipeline([
    ('Scale', StandardScaler()),
    ('model', SVC(class_weight='balanced', random_state=42))
])

# Hyperparameter combination
svc_grid = {
    'model__C'    : [0.1, 0.5, 1, 5, 10, 50],
    'model__gamma' : ['scale', 0.005, 0.01, 0.05, 0.1]
}

# Tune using 15 random combinations
rs_svc = RandomizedSearchCV(
    svc_pipe, svc_grid,
    n_iter=15, cv=cv,
    scoring='recall',
    random_state=42, n_jobs=-1
)

rs_svc.fit(x_train, y_train)

# Test the best SVM 
svc_pred = rs_svc.best_estimator_.predict(x_test)

svc_recall = recall_score(y_test , svc_pred)
svc_f1 = f1_score(y_test, svc_pred)

print("Best settings   :", rs_svc.best_params_)
print('CV recall       :', round(rs_svc.best_score_,3))
print("Test recall     :", round(svc_recall,3))
print('Test f1         :', round(svc_f1,3))
print('Optimism gap    :', round(rs_svc.best_score_ - svc_recall,3))
print('Flagged per 100 :', round(svc_pred.mean() * 100,1))

# compare with tuned Random forest 
forest_recall = recall_score(y_test,pred)
forest_gap = gs_recall.best_score_ - forest_recall

print('forest optimism gap:', round(forest_gap,3))

Best settings   : {'model__gamma': 'scale', 'model__C': 0.1}
CV recall       : 0.766
Test recall     : 0.673
Test f1         : 0.507
Optimism gap    : 0.094
Flagged per 100 : 47.9
forest optimism gap: 0.084


buiId the finaI ranking tabIe



In [ ]:
def evaluate(name, est):
    res = cross_validate(
        est, x_train, y_train,
        cv=cv,
        scoring=["recall", "f1"]
    )
    p = clone(est).fit(x_train, y_train).predict(x_test)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    return {
        "model": name,
        "cv_recall": round(res["test_recall"].mean(), 3),
        "cv_std": round(res["test_recall"].std(), 3),
        "cv_f1": round(res["test_f1"].mean(), 3),
        "test_recall": round(recall_score(y_test, p), 3),
        "test_f1": round(f1_score(y_test, p), 3),
        "found": tp,
        "missed": fn,
        "false_alarms": fp
    }

results = pd.DataFrame([
    evaluate("RF - Week 6", make_model(*candidates["RF balanced"])),
    evaluate("RF - tuned", gs_recall.best_estimator_),
    evaluate("SVC - Week 6", make_model(*candidates["SVC balanced"])),
    evaluate("SVC - tuned", rs_svc.best_estimator_)
]).sort_values("cv_recall", ascending=False)

results["clears_target"] = (
    (results["test_recall"] >= TARGET_RECALL) &
    (results["test_f1"] >= GUARD_F1)
)

print(results.to_string(index=False))

       model  cv_recall  cv_std  cv_f1  test_recall  test_f1  found  missed  false_alarms  clears_target
 SVC - tuned      0.766   0.038  0.556        0.673    0.507     37      18            54          False
  RF - tuned      0.739   0.041  0.541        0.745    0.536     41      14            57           True
SVC - Week 6      0.721   0.046  0.559        0.618    0.500     34      21            47          False
 RF - Week 6      0.652   0.051  0.521        0.655    0.545     36      19            41          False


1. The Model

The tuned Random Forest had the highest test recall at 0.745.

Searched on recall: ('class weight: 'balanced', 'max depth': 3, 'min samples leaf: 1) Test recall: 0.745 | Test F1: 0.536 |Patients found:41

2. The Evidence

cross validation recall std, test recall and test f1,against the week 5 target

3. What It Beat

becuase svc test recall is 0.673 that not true condition. then we have choose the RF tuned all condition true

4. In Patients

finds 41 of 55 misses 14,call back 57 healthy patients.

save the finaI settings

In [ ]:
final_settings = {
    "model":         "RandomForestClassifier",
    "params":        gs_recall.best_params_,
    "random_state":  42,
    "threshold":     0.5,
    "features":      list(x.columns),
    "split":         "test_size=0.2,random_state=42,stratify=y"
}
for k, v in final_settings.items():
    print(f"{k:13s} {v}")
results.to_csv(r"C:\Users\USER\Desktop\5thsem_OJT\project1_diabeties\reports\project_diebetes_raw_p1w7v2_results.csv", index=False)
print("\nsaved: project_diebetes_raw_p1w7v2_results.csv")

model         RandomForestClassifier
params        {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}
random_state  42
threshold     0.5
features      ['age', 'pregnancies', 'glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi', 'diabetes_pedigree', 'bmi_category_Obese', 'bmi_category_Over-weight', 'bmi_category_Under-weight ', 'glucose_band_Normal ', 'glucose_band_Prediabetic']
split         test_size=0.2,random_state=42,stratify=y

saved: project_diebetes_raw_p1w7v2_results.csv
